# TCC Data Preparation
## Stage 1: Download, process, and store the pouring dataset

This notebook downloads the multiview pouring dataset from HuggingFace,
extracts frames from videos, and stores the processed data to the
configured storage backend (local disk or S3).

**Config:** Edit `configs/pouring.yaml` to switch between local and S3 storage.

**Usage:**
```bash
# Local execution
make data-prep

# With papermill parameters
make data-prep NB_PARAMS="-p config_path configs/pouring.yaml"
```

In [ ]:
import sys
import os
import pathlib
import subprocess

IN_COLAB = "google.colab" in sys.modules

print("Python:", sys.version)
print("Running in Colab:", IN_COLAB)

In [ ]:
if IN_COLAB:
    REPO_DIR = pathlib.Path("tcc")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/aegean-ai/tcc", str(REPO_DIR)], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./tcc[notebooks]"], check=True)
else:
    print("Skipping clone/install \u2014 running inside dev container.")

In [ ]:
# \u2500\u2500 Papermill parameters \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
config_path = "configs/pouring.yaml"
fps = 15
image_width = 224
image_height = 224

In [ ]:
from tcc.storage import load_storage_config

storage = load_storage_config(config_path)

print(f"Storage backend: {storage.storage_backend}")
print(f"Dataset name:    {storage.dataset_name}")
print(f"Raw dir:         {storage.raw_dir}")
print(f"Processed dir:   {storage.processed_dir}")
if storage.cache_dir:
    print(f"Cache dir:       {storage.cache_dir}")

In [ ]:
if storage.storage_backend == "local":
    RAW_DIR = pathlib.Path(storage.raw_dir)
    PROCESSED_DIR = pathlib.Path(storage.processed_dir)
else:
    # S3 mode: use cache_dir for local staging
    RAW_DIR = pathlib.Path(storage.cache_dir) / "raw"
    PROCESSED_DIR = pathlib.Path(storage.cache_dir) / "processed" / storage.dataset_name

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"Local raw dir:       {RAW_DIR.resolve()}")
print(f"Local processed dir: {PROCESSED_DIR.resolve()}")

## 1. Download raw data from HuggingFace

The multiview pouring dataset is at
[`sermanet/multiview-pouring`](https://huggingface.co/datasets/sermanet/multiview-pouring).

In [ ]:
from huggingface_hub import snapshot_download

hf_cache_path = snapshot_download(
    repo_id="sermanet/multiview-pouring",
    repo_type="dataset",
    local_dir=str(RAW_DIR),
)

print("Dataset downloaded to:", hf_cache_path)

for split_dir in sorted(RAW_DIR.iterdir()):
    if split_dir.is_dir() and not split_dir.name.startswith("."):
        tfrecords = list(split_dir.glob("*.tfrecord*"))
        print(f"  {split_dir.name}/: {len(tfrecords)} TFRecord file(s)")

## 2. Convert videos to image-folder layout

Extract frames from `.mov` videos into the directory structure expected
by `tcc.datasets.VideoDataset`:

```
processed_dir/
\u2514\u2500\u2500 pouring/
    \u251c\u2500\u2500 train/video_001/frame_0000.png
    \u2514\u2500\u2500 val/video_050/frame_0000.png
```

In [ ]:
VIDEO_INPUT_DIR = RAW_DIR / "videos"
print(f"Converting videos from {VIDEO_INPUT_DIR}...")
print(f"Output: {PROCESSED_DIR}")

subprocess.run(
    [sys.executable, "-m", "tcc.dataset_preparation.videos_to_dataset",
     "--input-dir", str(VIDEO_INPUT_DIR),
     "--output-dir", str(PROCESSED_DIR.parent),
     "--name", storage.dataset_name,
     "--fps", str(fps),
     "--width", str(image_width),
     "--height", str(image_height),
     "--file-pattern", "**/*.mov",
     "--rotate"],
    check=True)

total = sum(len(f) for _, _, f in os.walk(PROCESSED_DIR))
print(f"Conversion complete: {total} files")

## 3. Upload to S3 lakehouse

Upload both raw and processed data to the MinIO lakehouse so that future
runs can use `storage_backend=s3` to skip download and conversion.

Requires `MINIO_ACCESS_KEY` and `MINIO_SECRET_KEY` in the environment.

In [ ]:
# Upload to S3 if credentials are available (regardless of storage_backend).
# This populates the lakehouse for future s3-mode runs.

_s3_key = os.environ.get("MINIO_ACCESS_KEY", "")
_s3_secret = os.environ.get("MINIO_SECRET_KEY", "")
_s3_endpoint = os.environ.get("MINIO_ENDPOINT", "http://192.168.1.26:9000")

if _s3_key and _s3_secret:
    import boto3
    from botocore.config import Config as BotoConfig

    s3 = boto3.client(
        "s3",
        endpoint_url=_s3_endpoint,
        aws_access_key_id=_s3_key,
        aws_secret_access_key=_s3_secret,
        config=BotoConfig(signature_version="s3v4"),
        region_name="us-east-1",
        use_ssl=_s3_endpoint.startswith("https"),
        verify=_s3_endpoint.startswith("https"),
    )

    S3_BUCKET = "landing"
    S3_PREFIX = "tcc/pouring"

    # ── Upload raw data (skip if already on S3) ──────────────────
    existing_raw = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"{S3_PREFIX}/raw/", MaxKeys=1)
    if existing_raw.get("KeyCount", 0) > 0:
        print(f"Raw data already exists at s3://{S3_BUCKET}/{S3_PREFIX}/raw/ — skipping upload")
    elif RAW_DIR.exists():
        raw_count = 0
        for dirpath, dirnames, filenames in os.walk(RAW_DIR):
            for fname in filenames:
                local_path = os.path.join(dirpath, fname)
                rel_path = os.path.relpath(local_path, RAW_DIR)
                s3_key = f"{S3_PREFIX}/raw/{rel_path}"
                s3.upload_file(local_path, S3_BUCKET, s3_key)
                raw_count += 1
                if raw_count % 100 == 0:
                    print(f"  raw: {raw_count} files uploaded...")
        print(f"Uploaded {raw_count} raw files to s3://{S3_BUCKET}/{S3_PREFIX}/raw/")
    else:
        print("No raw data to upload")

    # ── Upload processed data ────────────────────────────────────
    if PROCESSED_DIR.exists():
        proc_count = 0
        for dirpath, dirnames, filenames in os.walk(PROCESSED_DIR):
            for fname in filenames:
                local_path = os.path.join(dirpath, fname)
                rel_path = os.path.relpath(local_path, PROCESSED_DIR)
                s3_key = f"{S3_PREFIX}/processed/{rel_path}"
                s3.upload_file(local_path, S3_BUCKET, s3_key)
                proc_count += 1
                if proc_count % 100 == 0:
                    print(f"  processed: {proc_count} files uploaded...")
        print(f"Uploaded {proc_count} processed files to s3://{S3_BUCKET}/{S3_PREFIX}/processed/")
    else:
        print("No processed data to upload — run conversion first")
else:
    print("S3 credentials not set — skipping lakehouse upload.")
    print("Set MINIO_ACCESS_KEY and MINIO_SECRET_KEY to enable.")

In [ ]:
print("\n=== Data preparation complete ===")
print(f"Backend:       {storage.storage_backend}")
print(f"Processed at:  {storage.processed_dir}")
print("\nRun the training notebook next:")
print("  make train")